In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [4]:
def one_step_forecast(df, window):
    '''
    This function forecasts only one step forward

    df: dataframe for forecast
    window: number of steps or observation backward used to forecast one step forward
    '''
    d = df.values
    x = []
    n = len(df)
    idx = df.index[:-window]

    for start in range(n-window):
        end = start + window
        x.append(d[start:end])

    cols = [f'x_{i}' for i in range(1, window+1)]
    x = np.array(x).reshape(n-window, -1)
    y = df.iloc[window:].values
    
    df_xs = pd.DataFrame(x, columns=cols, index=idx)
    df_y = pd.DataFrame(y.reshape(-1), columns=['y'], index=idx)

    return pd.concat([df_y, df_xs], axis=1).dropna()

In [1]:
class Standardize:
    def __init__(self, df, split=0.10):
        self.data = df
        self.split = split

    def split_data(self):
        n = int(len(self.data) * self.split)
        train, test = self.data.iloc[:-n], self.data.iloc[-n:]
        n = int(len(train) * self.split)
        train, val = train.iloc[:-n], train.iloc[-n:]
        assert len(test) + len(train) + len(val) == len(self.data)

        return train, test, val

    def _transform(self, data):
        data_s = (data - self.mu) / self.sigma

        return data_s

    def fit_transform(self):
        train, test, val = self.split_data()
        self.mu, self.sigma = train.mean(), train.std()
        train_s = self._transform(train)
        test_s = self._transform(test)
        val_s = self._transform(val)

        return train_s, test_s, val_s

    def inverse(self, data):
        return (data * self.sigma) + self.mu

    def inverse_y(self, data):
        return (data * self.sigma[-1]) + self.mu[-1]

In [5]:
path = Path('../../../Code in Zip/Time-Series-Analysis-with-Python-Cookbook-main/datasets/Ch12')
energy = pd.read_csv(path.joinpath('energy_consumption.csv'), index_col='Month', parse_dates=True)
energy.columns = ['y']
energy.index.freq = 'MS'
en_df = one_step_forecast(energy, 10)

In [6]:
scale_energy = Standardize(en_df)
train_energy, test_energy, val_energy = scale_energy.fit_transform()